# Layouts and Tuple Morphisms

This notebook is a companion to the paper *Categorical Foundations for CuTe Layouts* (Colfax Research). It introduces the two central objects of the paper's flat theory:

1. **CuTe flat layouts** $(s_1,\dots,s_m):(d_1,\dots,d_m)$ and the coordinate $\to$ index functions they define, demonstrated with NVIDIA's `pycute` reference implementation; and
2. **tuple morphisms** — the morphisms of the category $\text{Tuple}$ — which give a categorical encoding of the *tractable* layouts, implemented in the `tract` package.

The bridge between the two worlds is the pair of translations

$$f \;\longmapsto\; L_f \qquad\text{and}\qquad L \;\longmapsto\; f_L \;(\text{the standard representation}),$$

which we construct at the end of the notebook and verify to be mutually inverse on tractable layouts.

## Setup

We import the public API of `tract` together with its pycute backend (`tract.backends.pycute`), which converts between tuple morphisms and `pycute.Layout` objects.

In [1]:
from pycute import Layout, size, idx2crd

from tract import TupleMorphism
from tract.backends import pycute as backend

## Flat layouts

A **flat layout** is a pair of integer tuples of common length written

$$L = (s_1,\dots,s_m):(d_1,\dots,d_m),$$

where $(s_1,\dots,s_m)$ is the **shape** and $(d_1,\dots,d_m)$ is the **stride**. In GPU programming, a layout mediates between the multi-dimensional *logical* coordinates of data and the one-dimensional *physical* addresses of memory: the layout defines the function

$$\ell_L(c_1,\dots,c_m) = \sum_{i=1}^{m} c_i\, d_i, \qquad 0 \le c_i < s_i,$$

sending a coordinate to a linear index. In `pycute` we construct a layout as `Layout(shape, stride)` and evaluate its layout function by calling it on a coordinate.

In [2]:
L = Layout((4, 2), (1, 4))
print("L =", L)
print("L(2, 1) =", L((2, 1)))   # 2*1 + 1*4

L = (4, 2):(1, 4)
L(2, 1) = 6


A layout of size $N = s_1 \cdots s_m$ also accepts a single linear index $0 \le x < N$: the index is first unpacked to the coordinate with the *colexicographic* (column-major) order, and the layout function is then applied. Tabulating $\ell_L$ over all indices displays the layout as a function $[0, N) \to \mathbb{Z}_{\ge 0}$.

In [3]:
for x in range(size(L)):
    coord = idx2crd(x, L.shape)
    print(f"x = {x}  ->  coord = {coord}  ->  index = {L(x)}")

x = 0  ->  coord = (0, 0)  ->  index = 0
x = 1  ->  coord = (1, 0)  ->  index = 1
x = 2  ->  coord = (2, 0)  ->  index = 2
x = 3  ->  coord = (3, 0)  ->  index = 3
x = 4  ->  coord = (0, 1)  ->  index = 4
x = 5  ->  coord = (1, 1)  ->  index = 5
x = 6  ->  coord = (2, 1)  ->  index = 6
x = 7  ->  coord = (3, 1)  ->  index = 7


With shape $(4,2)$ and stride $(1,4)$ the layout enumerates memory contiguously — it is the column-major identification of a $4 \times 2$ grid with $[0,8)$. Strides are arbitrary, however: they may leave gaps, overlap, or even be zero. Here is a *strided* layout that skips through memory, and a *broadcast* layout whose zero stride repeats indices.

In [4]:
strided = Layout((2, 2), (1, 10))
broadcast = Layout((4, 2), (0, 1))
print("strided   =", strided, "->", [strided(x) for x in range(size(strided))])
print("broadcast =", broadcast, "->", [broadcast(x) for x in range(size(broadcast))])

strided   = (2, 2):(1, 10) -> [0, 1, 10, 11]
broadcast = (4, 2):(0, 1) -> [0, 0, 0, 0, 1, 1, 1, 1]


## The category $\text{Tuple}$

The paper organizes (tractable) flat layouts into a category. The objects of $\text{Tuple}$ are finite tuples of positive integers $S = (s_1,\dots,s_m)$. A **tuple morphism**

$$f : (s_1,\dots,s_m) \longrightarrow (t_1,\dots,t_n)$$

*lying over* a map $\alpha : \langle m \rangle_* \to \langle n \rangle_*$ of pointed finite sets is required to satisfy $s_i = t_{\alpha(i)}$ whenever $\alpha(i) \ne *$. In code, $\alpha$ is the tuple `map` with **1-based** entries: `map[i] = j` says domain mode $i{+}1$ is matched with codomain mode $j$, while the value **0 denotes the basepoint** $*$ (the mode is collapsed).

We construct a tuple morphism as `TupleMorphism(domain, codomain, map)`.

In [5]:
f = TupleMorphism(domain=(4, 4), codomain=(4, 2, 4), map=(1, 3))
print("f =", f)

f = (4, 4) --(1, 3)--> (4, 2, 4)


The morphism $f$ above sends the first domain mode to codomain mode 1 and the second to codomain mode 3; codomain mode 2 is not hit. The constructor validates the defining condition $s_i = t_{\alpha(i)}$:

In [6]:
try:
    TupleMorphism(domain=(4, 4), codomain=(4, 2, 4), map=(1, 2))
except ValueError as e:
    print("ValueError:", e)

ValueError: Must satisfy s_i = t_α(i) for all i


A basepoint entry `0` in the map is always legal, regardless of the domain entry — the corresponding mode carries no constraint. The **size** and **cosize** of $f$ are the products of the domain and codomain entries.

In [7]:
g = TupleMorphism(domain=(3, 4), codomain=(4, 2, 4), map=(0, 1))
print("g =", g)
print("size(f) =", f.size(), "  cosize(f) =", f.cosize())
print("size(g) =", g.size(), "  cosize(g) =", g.cosize())

g = (3, 4) --(0, 1)--> (4, 2, 4)
size(f) = 16   cosize(f) = 32
size(g) = 12   cosize(g) = 32


## The layout $L_f$ of a tuple morphism

A tuple morphism $f : (s_1,\dots,s_m) \to (t_1,\dots,t_n)$ over $\alpha$ **encodes** a flat layout $L_f$ whose shape is the domain of $f$ and whose strides are *prefix products of the codomain*:

$$L_f = (s_1,\dots,s_m):(d_1,\dots,d_m), \qquad d_i = \begin{cases} t_1 t_2 \cdots t_{\alpha(i)-1} & \alpha(i) \ne * \\ 0 & \alpha(i) = *. \end{cases}$$

Intuitively, the codomain $(t_1,\dots,t_n)$ describes a factorization of an ambient linear memory of size $t_1\cdots t_n$ into nested blocks, and $\alpha$ places each domain mode at one level of that factorization; the stride of a mode is the size of everything below the level it occupies. The pycute backend computes this with `compute_flat_layout` (and `compute_flat_layout_components` for the raw shape/stride pair).

In [8]:
shape, stride = backend.compute_flat_layout_components(f)
print("f   =", f)
print("shape  =", shape)
print("stride =", stride)
print("L_f =", backend.compute_flat_layout(f))

f   = (4, 4) --(1, 3)--> (4, 2, 4)
shape  = (4, 4)
stride = (1, 8)
L_f = (4, 4):(1, 8)


Check the strides by hand: $\alpha(1) = 1$ gives the empty prefix product $d_1 = 1$, and $\alpha(2) = 3$ gives $d_2 = t_1 t_2 = 4 \cdot 2 = 8$. Basepoint modes get stride $0$:

In [9]:
print("g   =", g)
print("L_g =", backend.compute_flat_layout(g))

g   = (3, 4) --(0, 1)--> (4, 2, 4)
L_g = (3, 4):(0, 1)


The layout function of $L_f$ can equally be computed from the morphism itself: a coordinate $(c_1, c_2)$ of the domain is placed into the ambient space $4 \cdot 2 \cdot 4 = 32$ at position $c_1 \cdot 1 + c_2 \cdot 8$. We verify the two descriptions agree.

In [10]:
L_f = backend.compute_flat_layout(f)
values = [L_f(x) for x in range(size(L_f))]
manual = [c1 * 1 + c2 * 8 for c2 in range(4) for c1 in range(4)]
print("via pycute:", values)
print("by hand:   ", manual)
print("agree:", values == manual)

via pycute: [0, 1, 2, 3, 8, 9, 10, 11, 16, 17, 18, 19, 24, 25, 26, 27]
by hand:    [0, 1, 2, 3, 8, 9, 10, 11, 16, 17, 18, 19, 24, 25, 26, 27]
agree: True


## Tractability

Not every flat layout arises as $L_f$ for a tuple morphism: the strides of $L_f$ are, by construction, prefix products of a single tuple, so they must fit into a common multiplicative chain. This is captured by the notion of **tractability**.

Sort the modes $(s_i, d_i)$ of $L$ by stride (breaking ties by shape) and discard zero strides. Then $L$ is **tractable** if, in the sorted order, each product $s_i d_i$ divides the next stride $d_{i+1}$ — i.e. the sorted strides form a divisibility chain

$$d_1 \mid s_1 d_1 \mid d_2 \mid s_2 d_2 \mid d_3 \mid \cdots$$

The backend tests this with `is_tractable`.

In [11]:
A = Layout((2, 2, 2), (1, 2, 4))
B = Layout((2, 2), (1, 3))
print(f"A = {A}   tractable: {backend.is_tractable(A)}")
print(f"B = {B}   tractable: {backend.is_tractable(B)}")

A = (2, 2, 2):(1, 2, 4)   tractable: True
B = (2, 2):(1, 3)   tractable: False


The layout $A = (2,2,2):(1,2,4)$ is tractable: its sorted strides satisfy $2\cdot 1 \mid 2$ and $2 \cdot 2 \mid 4$. The layout $B = (2,2):(1,3)$ is **not** tractable: the first mode occupies $s_1 d_1 = 2 \cdot 1 = 2$ slots, but the next stride is $3$, and $2 \nmid 3$ — there is no way to factor the ambient memory into blocks so that both modes sit at whole levels.

Tractability does not require compactness. The strided layout $(2,2):(1,4)$ leaves a gap, yet is tractable ($2 \cdot 1 \mid 4$), and sorting handles modes given in any order:

In [12]:
for shape, stride in [((2, 2), (1, 4)), ((4, 8, 4), (128, 1, 16)), ((2, 2, 2), (1, 7, 4))]:
    L_ = Layout(shape, stride)
    print(f"{str(L_):<22} tractable: {backend.is_tractable(L_)}")

(2, 2):(1, 4)          tractable: True
(4, 8, 4):(128, 1, 16) tractable: True
(2, 2, 2):(1, 7, 4)    tractable: False


## The standard representation $f_L$

Every tractable flat layout $L$ has a **standard representation**: a canonical tuple morphism $f_L$ with $L_{f_L} = L$. Its codomain interleaves the sorted strides with the shapes — recording both the blocks the layout occupies and the gaps it skips — and its map places each domain mode at its block. The backend computes it with `compute_Tuple_morphism` (raising `ValueError` on a non-tractable layout).

In [13]:
f_A = backend.compute_Tuple_morphism(A)
print("A   =", A)
print("f_A =", f_A)

A   = (2, 2, 2):(1, 2, 4)
f_A = (2, 2, 2) --(1, 2, 3)--> (2, 2, 2)


In [14]:
C = Layout((2, 2), (1, 4))
f_C = backend.compute_Tuple_morphism(C)
print("C   =", C)
print("f_C =", f_C)

C   = (2, 2):(1, 4)
f_C = (2, 2) --(1, 3)--> (2, 2, 2)


In `f_C` the codomain $(2,2,2)$ factors the ambient memory of size $8$; the map $(1,3)$ skips codomain mode 2, which is exactly the gap of size $2$ between the two modes of $C = (2,2):(1,4)$.

Attempting the standard representation of a non-tractable layout fails:

In [15]:
try:
    backend.compute_Tuple_morphism(B)
except ValueError as e:
    print("ValueError:", e)

ValueError: The provided layout is not tractable.


## The round trip $L \mapsto f_L \mapsto L_{f_L}$

The two translations are mutually inverse on tractable layouts: computing the layout of the standard representation recovers the layout we started with. We check this with `layouts_agree`, which compares layouts mode-by-mode after flattening (nullifying strides of size-1 modes, which are invisible to the layout function).

In [16]:
for L_ in [A, C, Layout((4, 8, 4), (128, 1, 16))]:
    f_L = backend.compute_Tuple_morphism(L_)
    round_trip = backend.compute_flat_layout(f_L)
    print(f"L = {str(L_):<22} f_L = {str(f_L):<42} L_(f_L) = {round_trip}")
    assert backend.layouts_agree(round_trip, L_)
print("round trip recovers every layout: True")

L = (2, 2, 2):(1, 2, 4)    f_L = (2, 2, 2) --(1, 2, 3)--> (2, 2, 2)         L_(f_L) = (2, 2, 2):(1, 2, 4)
L = (2, 2):(1, 4)          f_L = (2, 2) --(1, 3)--> (2, 2, 2)               L_(f_L) = (2, 2):(1, 4)
L = (4, 8, 4):(128, 1, 16) f_L = (4, 8, 4) --(5, 1, 3)--> (8, 2, 4, 2, 4)   L_(f_L) = (4, 8, 4):(128, 1, 16)
round trip recovers every layout: True


Conversely, the layout $L_f$ of *any* tuple morphism is tractable — its strides are prefix products of the codomain by construction. So the tuple morphisms are precisely a categorical bookkeeping of the tractable layouts:

In [17]:
for h in [f, g, TupleMorphism((8, 3), (3, 4, 8), (3, 1))]:
    L_h = backend.compute_flat_layout(h)
    print(f"{str(h):<35} L_h = {str(L_h):<18} tractable: {backend.is_tractable(L_h)}")

(4, 4) --(1, 3)--> (4, 2, 4)        L_h = (4, 4):(1, 8)      tractable: True
(3, 4) --(0, 1)--> (4, 2, 4)        L_h = (3, 4):(0, 1)      tractable: True
(8, 3) --(3, 1)--> (3, 4, 8)        L_h = (8, 3):(12, 1)     tractable: True


## Summary

* A flat layout $(s_1,\dots,s_m):(d_1,\dots,d_m)$ defines the coordinate $\to$ index function $\ell_L(c) = \sum_i c_i d_i$.
* A tuple morphism $f : S \to T$ over $\alpha$ (1-based map, $0$ = basepoint) encodes the layout $L_f$ with shape $S$ and strides the prefix products $t_1 \cdots t_{\alpha(i)-1}$.
* A layout is **tractable** exactly when its sorted strides form a divisibility chain — equivalently, when it is $L_f$ for some tuple morphism — and $(2,2):(1,3)$ is the smallest kind of failure ($3 \nmid 2 \cdot 1$).
* `compute_Tuple_morphism` produces the standard representation $f_L$, and $L \mapsto f_L \mapsto L_{f_L}$ is the identity on tractable layouts, up to trivial modes.

The next notebook, `02_operations.ipynb`, develops the *algebra* of tuple morphisms — composition, coalescence, complements, division and product — and cross-validates each operation against the CuTe layout algebra.